# Asignación de coordenadas – Método 2 (KDE + DBSCAN)
Genera un archivo de coordenadas únicas con su asignación a hubs M2 a 50 / 100 / 150 millas.
- **Input hubs**: `../5. merge dbscan/results/final_results.csv`
- **Input cargas**: `../2. cleaning_loads/parquetloadsfull.parquet`
- **Output**: `coords_assign_m2.parquet`

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from tqdm.auto import tqdm

/home/karen/hubs/jupyter-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Hubs M2

In [14]:
hubs_raw = pd.read_csv('../5. merge dbscan/results/final_results.csv')

rows = []
for i, (_, r) in enumerate(hubs_raw.dropna(subset=['lat', 'lng']).iterrows()):
    if pd.notna(r.get('new_name')):
        hub_id = str(r['id'])
    else:
        hub_id = f'C{i}' if pd.notna(r.get('cluster')) else None
    if hub_id is not None:
        rows.append({
            'hub_id': hub_id,
            'hub_name': r.get('new_name') or hub_id,
            'lat': float(r['lat']),
            'lng': float(r['lng']),
        })

hubs = pd.DataFrame(rows).reset_index(drop=True)
print(f'Hubs M2: {len(hubs)}')

Hubs M2: 192


## 2. Coordenadas únicas

In [15]:
raw = (
    pl.scan_parquet('../2. cleaning_loads/parquetloadsfull.parquet')
    .select(['LatOrigin', 'LngOrigin', 'LatDestination', 'LngDestination'])
    .collect()
)

orig = raw.select([
    pl.col('LatOrigin').cast(pl.Float64).round(6).alias('lat'),
    pl.col('LngOrigin').cast(pl.Float64).round(6).alias('lng'),
]).drop_nulls()

dest = raw.select([
    pl.col('LatDestination').cast(pl.Float64).round(6).alias('lat'),
    pl.col('LngDestination').cast(pl.Float64).round(6).alias('lng'),
]).drop_nulls()

coords = (
    pl.concat([orig, dest])
    .unique()
    .to_pandas()
    .reset_index(drop=True)
)

del raw, orig, dest
print(f'Coordenadas unicas: {len(coords):,}')

Coordenadas unicas: 738,827


## 3. Asignación vectorizada por batches

In [16]:
def assign_batched(coords_df, hubs_df, batch_size=5_000, radii=(50, 100, 150)):
    n        = len(coords_df)
    hub_lats = np.radians(hubs_df['lat'].values)
    hub_lngs = np.radians(hubs_df['lng'].values)
    hub_ids  = hubs_df['hub_id'].values

    near_hub = np.empty(n, dtype=object)
    hubs_r   = {r: [None] * n for r in radii}

    for start in tqdm(range(0, n, batch_size)):
        end  = min(start + batch_size, n)
        blat = np.radians(coords_df['lat'].values[start:end, np.newaxis])
        blng = np.radians(coords_df['lng'].values[start:end, np.newaxis])

        dlat = hub_lats - blat
        dlng = hub_lngs - blng
        a    = (np.sin(dlat / 2) ** 2
                + np.cos(blat) * np.cos(hub_lats) * np.sin(dlng / 2) ** 2)
        dist = 3958.8 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

        near_hub[start:end] = hub_ids[dist.argmin(axis=1)]
        for r in radii:
            mask = dist <= r
            for i in range(end - start):
                hubs_r[r][start + i] = list(hub_ids[mask[i]])

    out = coords_df.copy()
    out['near_hub_m2'] = near_hub
    for r in radii:
        out[f'hubs_{r}m_m2'] = hubs_r[r]
    return out


In [17]:
coords_assigned = assign_batched(coords, hubs)
coords_assigned.head()

100%|████████████████████████████████████████████████████████████████████████████████| 148/148 [00:25<00:00,  5.71it/s]


,lat,lng,near_hub_m2,hubs_50m_m2,hubs_100m_m2,hubs_150m_m2
0,32.430800,-99.643900,C168,[],"[C168, C169]","[C168, C169]"
1,40.047940,-76.255710,140.0,"[18.0, 140.0]","[18.0, 82.0, 116.0, 137.0, 140.0]","[18.0, 82.0, 116.0, 119.0, 137.0, 140.0, 202.0..."
2,46.309580,-119.291570,C184,[C184],[C184],"[C113, C182, C184, C185]"
3,46.898291,-96.852649,94.0,[94.0],"[94.0, C127]","[94.0, C127, C143, C144, C167]"
4,27.587536,-82.425094,70.0,[70.0],"[68.0, 70.0, C106]","[68.0, 70.0, C105, C106]"


## 4. Guardar

In [18]:
out = pl.from_pandas(coords_assigned)
out.write_parquet('coords_assign_m2.parquet')
print(f'Guardado: coords_assign_m2.parquet  ({len(out):,} filas)')
print(out.schema)
print(out.head())

Guardado: coords_assign_m2.parquet  (738,827 filas)
Schema([('lat', Float64), ('lng', Float64), ('near_hub_m2', String), ('hubs_50m_m2', List(String)), ('hubs_100m_m2', List(String)), ('hubs_150m_m2', List(String))])
shape: (5, 6)
┌───────────┬────────────┬─────────────┬───────────────────┬───────────────────┬───────────────────┐
│ lat       ┆ lng        ┆ near_hub_m2 ┆ hubs_50m_m2       ┆ hubs_100m_m2      ┆ hubs_150m_m2      │
│ ---       ┆ ---        ┆ ---         ┆ ---               ┆ ---               ┆ ---               │
│ f64       ┆ f64        ┆ str         ┆ list[str]         ┆ list[str]         ┆ list[str]         │
╞═══════════╪════════════╪═════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ 32.4308   ┆ -99.6439   ┆ C168        ┆ []                ┆ ["C168", "C169"]  ┆ ["C168", "C169"]  │
│ 40.04794  ┆ -76.25571  ┆ 140.0       ┆ ["18.0", "140.0"] ┆ ["18.0", "82.0",  ┆ ["18.0", "82.0",  │
│           ┆            ┆             ┆                   ┆ …